#  Character Embeddings with Keras

Steps:
1. Preprocess texts: lowercase and map characters to integer IDs (via `TextVectorization` at character level).
2. Train an embedding model in Keras on the large text.
3. Apply the trained embedding to a smaller text and display embedding samples.
4. Discuss transferred information and explain OOV (out-of-vocabulary) handling.


In [1]:
import os, numpy as np, tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding, Input, Dense
from tensorflow.keras import Model
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)
np.random.seed(42)
print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.11.0


## 1.Load Large Text
We use the uploaded file as the large corpus.

In [2]:
large_text_path = "Question 1.txt"
with open(large_text_path, "r", encoding="utf-8", errors="ignore") as f:
    large_text = f.read()
print("Large text length (chars):", len(large_text))
print("Preview:\n", large_text[:500].replace("\n"," ")[:500], "...")

Large text length (chars): 8159
Preview:
 Millions join anti-Trump 'No Kings' protests across US 7 hours ago  Share  Save FREDERIC J. BROWN/AFP via Getty Images Many protesters and the big inflatable Donald Trump baby is hovering above themFREDERIC J. BROWN/AFP via Getty Images The Donald Trump "baby blimp", which has become a common sight at protests over the years, made an appearance in Los Angeles Grace Eliza GoodwinNew York andCaitlin WilsonWashington Huge crowds have taken part in "No Kings" protests against President Donald Trump' ...


## 2.Define a Smaller Text with Some New Characters
We inject a few characters/symbols likely **not** present in the large text (e.g., emojis, math symbols) to test OOV behavior.

In [3]:
smaller_text = (
    "Keras char embeddings demo! New chars: Ω≈ç√∫˜µ≤≥÷ — \n"
    "We also mix CASES and Symbols: @#&*()[]{}|;:'\",.<>?/\n"
    "Plus some digits: 0123456789\n"
)
print("Smaller text length (chars):", len(smaller_text))
print("Smaller text sample:\n", smaller_text[:200])

Smaller text length (chars): 135
Smaller text sample:
 Keras char embeddings demo! New chars: Ω≈ç√∫˜µ≤≥÷ — 
We also mix CASES and Symbols: @#&*()[]{}|;:'",.<>?/
Plus some digits: 0123456789



## 3.Preprocessing with Character-Level Mapping
We lowercase and tokenize **by character** using `TextVectorization`. We include an OOV token so unseen characters are captured.

In [4]:
sequence_length = 128  
vectorizer = TextVectorization(
    standardize="lower",
    split="character",
    output_mode="int",
    output_sequence_length=sequence_length,
)

large_ds = tf.data.Dataset.from_tensor_slices([large_text])
vectorizer.adapt(large_ds)

vocab = vectorizer.get_vocabulary()
vocab_size = len(vocab)
print("Vocabulary size:", vocab_size)
print("First 30 vocab entries:", vocab[:30])

small_ids = vectorizer(tf.constant([smaller_text]))
print("Encoded smaller text shape:", small_ids.shape)
print("Encoded smaller text sample (first row, first 120 ids):\n", small_ids[0][:120].numpy())

Vocabulary size: 50
First 30 vocab entries: ['', '[UNK]', ' ', 'e', 'a', 't', 's', 'i', 'o', 'n', 'r', 'd', 'h', 'c', 'l', 'g', 'p', 'u', 'm', 'w', '\n', 'y', 'f', 'b', ',', 'v', '.', '"', 'k', "'"]
Encoded smaller text shape: (1, 128)
Encoded smaller text sample (first row, first 120 ids):
 [28  3 10  4  6  2 13 12  4 10  2  3 18 23  3 11 11  7  9 15  6  2 11  3
 18  8 49  2  9  3 19  2 13 12  4 10  6 42  2  1  1  1  1  1  1  1  1  1
  1  2  1  2 20 19  3  2  4 14  6  8  2 18  7 35  2 13  4  6  3  6  2  4
  9 11  2  6 21 18 23  8 14  6 42  2  1  1  1  1  1  1  1  1  1  1  1  1
 42 29 27 24 26  1  1 47 31 20 16 14 17  6  2  6  8 18  3  2 11  7 15  7]


## 4.Build Training Sequences from the Large Text
We create sliding windows of fixed length and predict the **next character** (simple language-model-like objective).

In [5]:
char2idx = {ch: i for i, ch in enumerate(vocab)}
def encode_chars(raw_text):
    raw_text = raw_text.lower()
    ids = []
    for ch in raw_text:
       
        ids.append(char2idx.get(ch, 1))
    return np.array(ids, dtype=np.int32)

large_ids_full = encode_chars(large_text)

def build_sequences(id_array, window, stride):
    X, y = [], []
    end = len(id_array) - window - 1
    for start in range(0, max(0, end), stride):
        X.append(id_array[start:start+window])
        y.append(id_array[start+1:start+window+1])
    return np.array(X, dtype=np.int32), np.array(y, dtype=np.int32)

X_train, y_train = build_sequences(large_ids_full, sequence_length, stride=64)
print("X_train shape:", X_train.shape, "y_train shape:", y_train.shape)

X_train shape: (126, 128) y_train shape: (126, 128)


## 5.Train a Character Embedding Model (Keras)
A lightweight model: **Embedding → TimeDistributed(Dense)** for next-character prediction. We train a few epochs for demonstration.

In [6]:
embed_dim = 32
inputs = Input(shape=(sequence_length,), dtype="int32")
x = Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True, name="char_embedding")(inputs)
logits = tf.keras.layers.TimeDistributed(Dense(vocab_size))(x)
model = Model(inputs, logits)
model.compile(optimizer="adam", loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True))
model.summary()

batch_size = 128
ds = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(2048).batch(batch_size).prefetch(tf.data.AUTOTUNE)
history = model.fit(ds, epochs=2, callbacks=[EarlyStopping(patience=1, restore_best_weights=True)], verbose=1)

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 128)]             0         
                                                                 
 char_embedding (Embedding)  (None, 128, 32)           1600      
                                                                 
 time_distributed (TimeDistr  (None, 128, 50)          1650      
 ibuted)                                                         
                                                                 
Total params: 3,250
Trainable params: 3,250
Non-trainable params: 0
_________________________________________________________________
Epoch 1/2
1/1 [==============================] - 1s 1s/step - loss: 3.9128
Epoch 2/2
1/1 [==============================] - 0s 32ms/step - loss: 3.9102


## 7.Apply the Trained Embedding to the Smaller Text and Show Samples

In [7]:
small_ids_full = encode_chars(smaller_text)
pad_len = max(0, sequence_length - len(small_ids_full))
if pad_len > 0:
    small_ids_window = np.pad(small_ids_full, (0, pad_len), constant_values=0)[:sequence_length]
else:
    small_ids_window = small_ids_full[:sequence_length]
small_ids_window = small_ids_window.reshape(1, -1)

embedding_layer = model.get_layer("char_embedding")
small_embeddings = embedding_layer(small_ids_window)  
print("Small embeddings shape:", small_embeddings.shape)

emb_slice = small_embeddings[0, :20].numpy()
print("First 20 embedded vectors (truncated to 5 dims):\n", np.round(emb_slice[:, :5], 4))

Small embeddings shape: (1, 128, 32)
First 20 embedded vectors (truncated to 5 dims):
 [[-0.0341  0.0149 -0.043   0.0171 -0.0424]
 [-0.0305 -0.0246  0.0085  0.027   0.0503]
 [ 0.0118 -0.0148  0.0434  0.0249 -0.0192]
 [ 0.0139  0.0098  0.0152  0.0476 -0.0107]
 [ 0.0138  0.0462 -0.0182 -0.0383  0.0041]
 [-0.0237  0.0158  0.0462  0.0477 -0.0256]
 [ 0.0295 -0.0246  0.0316 -0.0035  0.0442]
 [ 0.0291 -0.0299 -0.0287  0.0498 -0.0003]
 [ 0.0139  0.0098  0.0152  0.0476 -0.0107]
 [ 0.0118 -0.0148  0.0434  0.0249 -0.0192]
 [-0.0237  0.0158  0.0462  0.0477 -0.0256]
 [-0.0305 -0.0246  0.0085  0.027   0.0503]
 [-0.0422 -0.0021  0.0093 -0.0033  0.0186]
 [ 0.0236 -0.0249 -0.0445 -0.0117 -0.0304]
 [-0.0305 -0.0246  0.0085  0.027   0.0503]
 [-0.0227  0.0304 -0.0233  0.0127  0.01  ]
 [-0.0227  0.0304 -0.0233  0.0127  0.01  ]
 [-0.0014 -0.0424 -0.0238 -0.0078 -0.0301]
 [-0.0038  0.0375  0.0122 -0.0293  0.0312]
 [ 0.0177 -0.0106 -0.0152 -0.0352  0.045 ]]
